# MÓDULO 4
## Tema 3. Introducción a bases de datos con SQLite y Python

Este notebook está pensado para alumnado que **nunca ha trabajado con bases de datos**.

La progresión será deliberadamente gradual:

1. Qué es una base de datos.
2. Qué son tablas, filas y columnas.
3. Crear una base de datos SQLite desde Python.
4. Crear una primera tabla.
5. Insertar datos.
6. Consultar datos con `SELECT`.
7. Recuperar resultados con `fetchone()` y `fetchall()`.
8. Modificar y borrar datos.
9. Inspeccionar una base de datos desde Python.
10. Relacionar tablas con claves foráneas y `JOIN`.
11. Extras: índices, Pandas y buenas prácticas.


## Cómo usar este notebook

Ejecuta las celdas en orden. La base de datos de ejemplo se llamará:

```text
tienda_didactica.db
```

SQLite guarda toda la base de datos en un único fichero `.db`.

En este notebook se borra ese fichero al principio para que los ejemplos sean reproducibles. En una aplicación real no borraríamos la base de datos de esa forma.


## 1. Idea básica: tablas, filas y columnas

Una base de datos sirve para **guardar datos de forma organizada** y poder consultarlos después.

Una tabla se parece a una hoja de cálculo:

| id | email | total | status |
|---:|---|---:|---|
| 1 | ana@example.com | 19.99 | paid |
| 2 | luis@example.com | 55.00 | pending |

Conceptos básicos:

- **Tabla**: conjunto de datos de un mismo tipo, por ejemplo `pedidos`.
- **Columna**: un campo concreto, por ejemplo `email` o `total`.
- **Fila**: un registro completo.
- **Clave primaria**: columna que identifica una fila de forma única.


## 2. Qué es SQLite

SQLite es una base de datos ligera y embebida.

Ventajas:

- No necesita servidor.
- Vive en un fichero `.db`.
- Python incluye el módulo `sqlite3`.
- Es ideal para aprender, prototipos, herramientas internas y análisis local.

Limitación importante:

- No es la mejor opción para aplicaciones con muchísimas escrituras simultáneas de muchos usuarios.


## 3. Comprobar la versión de SQLite usada por Python

Python incluye el módulo `sqlite3` en la biblioteca estándar.

La versión más relevante es `sqlite3.sqlite_version`, que indica la versión de SQLite que está usando Python.


In [ ]:
import sqlite3

print("Versión de SQLite usada por Python:", sqlite3.sqlite_version)
print("Ruta del módulo sqlite3:", sqlite3.__file__)

## 4. Crear una base de datos

Para crear o abrir una base de datos usamos:

```python
sqlite3.connect("nombre.db")
```

Si el fichero no existe, SQLite lo crea. Si ya existe, lo abre.


In [ ]:
import sqlite3
from pathlib import Path

DB_PATH = Path("tienda_didactica.db")

# Para clase: empezamos siempre desde cero.
# En una aplicación real no borraríamos la base de datos así.
if DB_PATH.exists():
    DB_PATH.unlink()

conn = sqlite3.connect(DB_PATH)
conn.close()

print("Base de datos creada:", DB_PATH)
print("Existe el fichero:", DB_PATH.exists())

## 5. Usar `with` para abrir la conexión

Para hablar con la base de datos necesitamos una **conexión**.

Usaremos este patrón:

```python
with sqlite3.connect(DB_PATH) as conn:
    ...
```

Ventajas:

- Si todo va bien, los cambios se guardan.
- Si ocurre un error, los cambios se revierten.
- El código queda más limpio.


## 6. Crear nuestra primera tabla

Crearemos una tabla `pedidos` con estas columnas (los tipos estan explicados justo debajo en el siguiente punto):

| Columna | Tipo | Significado |
|---|---|---|
| `id` | `INTEGER PRIMARY KEY` | identificador único |
| `email` | `TEXT NOT NULL` | email del cliente |
| `total` | `REAL NOT NULL` | importe del pedido |
| `status` | `TEXT NOT NULL` | estado del pedido |
| `created_at` | `TEXT NOT NULL` | fecha/hora en texto |

La instrucción SQL para crear una tabla es `CREATE TABLE`.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute("""
        CREATE TABLE pedidos (
            id INTEGER PRIMARY KEY,
            email TEXT NOT NULL,
            total REAL NOT NULL,
            status TEXT NOT NULL,
            created_at TEXT NOT NULL
        );
    """)

print("Tabla pedidos creada")

## 7. Tipos de datos y restricciones básicas

Tipos habituales en SQLite:

| Tipo | Uso habitual |
|---|---|
| `INTEGER` | enteros, identificadores |
| `REAL` | números con decimales |
| `TEXT` | texto, emails, fechas en ISO |
| `BLOB` | datos binarios |
| `NULL` | ausencia de valor |

Restricciones útiles:

| Restricción | Qué garantiza |
|---|---|
| `PRIMARY KEY` | identifica cada fila |
| `NOT NULL` | el campo es obligatorio |
| `UNIQUE` | evita duplicados |
| `CHECK` | valida una condición |
| `DEFAULT` | define un valor por defecto |

De momento usamos pocas restricciones para no meter demasiadas cosas a la vez.


## 8. Insertar una fila con `INSERT INTO`

Para guardar datos usamos `INSERT INTO`.

Estructura SQL:

```sql
INSERT INTO tabla(columna1, columna2)
VALUES (valor1, valor2);
```

Desde Python usaremos `?` para pasar valores variables.
Eso se llama **consulta parametrizada**.


In [ ]:
from datetime import datetime

def now_iso():
    """Devuelve la fecha y hora actual en formato ISO legible."""
    return datetime.now().isoformat(timespec="seconds")

with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute(
        "INSERT INTO pedidos(email, total, status, created_at) VALUES (?, ?, ?, ?);",
        ("ana@example.com", 19.99, "paid", now_iso())
    )

print("Pedido insertado")


## 9. Por qué usamos `?`

Esta forma es correcta:

```python
cur.execute(
    "INSERT INTO pedidos(email, total) VALUES (?, ?);",
    (email, total)
)
```

Evita:

- problemas con comillas;
- errores de formato;
- vulnerabilidades como SQL injection.

Regla práctica: **los datos variables se pasan como parámetros, no se concatenan dentro del SQL**.


## 10. Insertar varias filas

Ahora añadiremos más pedidos para tener datos que consultar.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute(
        "INSERT INTO pedidos(email, total, status, created_at) VALUES (?, ?, ?, ?);",
        ("luis@example.com", 55.00, "pending", now_iso())
    )

    cur.execute(
        "INSERT INTO pedidos(email, total, status, created_at) VALUES (?, ?, ?, ?);",
        ("marta@example.com", 120.50, "paid", now_iso())
    )

    cur.execute(
        "INSERT INTO pedidos(email, total, status, created_at) VALUES (?, ?, ?, ?);",
        ("ana@example.com", 9.99, "cancelled", now_iso())
    )

print("Pedidos insertados")


# 11. Consultar datos con `SELECT`

Una base de datos no solo sirve para guardar datos. Lo normal es querer consultarlos.

La instrucción SQL para consultar datos es `SELECT`.

```sql
SELECT * FROM pedidos;
```

Significa:

- `SELECT`: quiero consultar datos.
- `*`: quiero todas las columnas.
- `FROM pedidos`: desde la tabla `pedidos`.


## 12. `execute` ejecuta la consulta, pero no muestra los datos

Cuando hacemos:

```python
cur.execute("SELECT * FROM pedidos;")
```

Python manda la consulta a SQLite.

Pero los datos no aparecen solos. Hay que recuperarlos con:

- `fetchone()`
- `fetchall()`
- o recorriendo el cursor con un `for`.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    resultado = cur.execute("SELECT * FROM pedidos;")

    print("Objeto devuelto por execute:")
    print(resultado)

    # Cerramos el cursor para liberar la consulta.
    # En la siguiente celda recuperaremos los datos correctamente.
    cur.close()

## 13. Recuperar todas las filas con `fetchall()`

`fetchall()` recupera **todas las filas** que devuelve la consulta.

Devuelve una lista de tuplas.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute("SELECT * FROM pedidos;")
    filas = cur.fetchall()

print(filas)


## 14. Entender las tuplas devueltas

Por defecto, cada fila viene como una tupla.

Si una fila es:

```python
(1, 'ana@example.com', 19.99, 'paid', '2026-05-05T12:00:00')
```

Entonces:

- `fila[0]` es el `id`;
- `fila[1]` es el `email`;
- `fila[2]` es el `total`;
- `fila[3]` es el `status`;
- `fila[4]` es `created_at`.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute("SELECT id, email, total, status, created_at FROM pedidos;")
    filas = cur.fetchall()

for fila in filas:
    print("ID:", fila[0])
    print("Email:", fila[1])
    print("Total:", fila[2])
    print("Estado:", fila[3])
    print("Fecha:", fila[4])
    print("---")


## 15. Consultar solo algunas columnas

No siempre necesitamos todas las columnas.

En vez de:

```sql
SELECT * FROM pedidos;
```

podemos pedir solo algunas:

```sql
SELECT id, email, total FROM pedidos;
```


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute("SELECT id, email, total FROM pedidos;")
    filas = cur.fetchall()

for fila in filas:
    print(fila)


## 16. Recuperar una sola fila con `fetchone()`

`fetchone()` recupera una única fila.

Es útil cuando esperamos un solo resultado, por ejemplo buscar por `id`.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute("SELECT id, email, total, status FROM pedidos WHERE id = ?;", (1,))
    fila = cur.fetchone()

print(fila)


## 17. Si `fetchone()` no encuentra nada, devuelve `None`

Cuando buscamos una fila concreta, conviene comprobar si se ha encontrado.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute("SELECT id, email, total, status FROM pedidos WHERE id = ?;", (999,))
    fila = cur.fetchone()

if fila is None:
    print("No existe ningún pedido con ese ID")
else:
    print("Pedido encontrado:", fila)


## 18. Filtrar con `WHERE`

`WHERE` sirve para consultar solo las filas que cumplen una condición.

```sql
SELECT id, email, total, status
FROM pedidos
WHERE status = 'paid';
```

En Python pasaremos el valor como parámetro.


In [ ]:
estado_buscado = "paid"

with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute(
        "SELECT id, email, total, status FROM pedidos WHERE status = ?;",
        (estado_buscado,)
    )
    filas = cur.fetchall()

print("Pedidos con estado", estado_buscado)
for fila in filas:
    print(fila)


### Nota sobre `(estado_buscado,)`

Los parámetros se pasan como una tupla.

Una tupla de un solo elemento necesita coma:

```python
(estado_buscado,)
```


## 19. Ordenar con `ORDER BY`

`ORDER BY` permite ordenar los resultados.

- `ASC`: ascendente.
- `DESC`: descendente.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute("""
        SELECT id, email, total, status
        FROM pedidos
        ORDER BY total DESC;
    """)
    filas = cur.fetchall()

for fila in filas:
    print(fila)


## 20. Limitar resultados con `LIMIT`

`LIMIT` permite quedarnos solo con un número máximo de filas.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute("""
        SELECT id, email, total, status
        FROM pedidos
        ORDER BY total DESC
        LIMIT 2;
    """)
    filas = cur.fetchall()

for fila in filas:
    print(fila)


## 21. Recorrer directamente el cursor

También podemos recorrer directamente el resultado sin llamar a `fetchall()`.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    for fila in cur.execute("SELECT id, email, total FROM pedidos ORDER BY id;"):
        print(fila)


## 22. Mini práctica 1

Antes de seguir, practica:

1. Inserta un nuevo pedido.
2. Consulta todos los pedidos.
3. Consulta solo los pedidos `paid`.
4. Consulta el pedido con `id = 2` usando `fetchone()`.
5. Ordena los pedidos de mayor a menor importe.


In [ ]:
# Espacio para practicar


# 23. Actualizar datos con `UPDATE`

`UPDATE` modifica filas existentes.

```sql
UPDATE tabla
SET columna = nuevo_valor
WHERE condicion;
```

La parte más importante es `WHERE`. Sin `WHERE`, modificarías todas las filas.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute(
        "UPDATE pedidos SET status = ? WHERE email = ?;",
        ("paid", "luis@example.com")
    )

    print("Filas modificadas:", cur.rowcount)


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute("SELECT id, email, total, status FROM pedidos ORDER BY id;")
    filas = cur.fetchall()

for fila in filas:
    print(fila)


# 24. Borrar datos con `DELETE`

`DELETE` elimina filas.

```sql
DELETE FROM tabla
WHERE condicion;
```

Sin `WHERE`, borrarías todas las filas de la tabla.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute(
        "DELETE FROM pedidos WHERE status = ?;",
        ("cancelled",)
    )

    print("Filas borradas:", cur.rowcount)


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute("SELECT id, email, total, status FROM pedidos ORDER BY id;")
    filas = cur.fetchall()

for fila in filas:
    print(fila)


## 25. CRUD: resumen

CRUD resume las cuatro operaciones básicas:

| Letra | Operación | SQL |
|---|---|---|
| C | Create | `INSERT INTO` |
| R | Read | `SELECT` |
| U | Update | `UPDATE` |
| D | Delete | `DELETE` |


# 26. Acceder a columnas por nombre con `sqlite3.Row`

Hasta ahora hemos usado tuplas:

```python
fila[0]
fila[1]
```

Para mejorar la legibilidad, podemos usar:

```python
conn.row_factory = sqlite3.Row
```

Así podremos acceder por nombre:

```python
fila["email"]
```


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()

    cur.execute("SELECT id, email, total, status FROM pedidos ORDER BY id;")
    filas = cur.fetchall()

for fila in filas:
    print("Email:", fila["email"], "| Total:", fila["total"], "| Estado:", fila["status"])


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row
    filas = conn.execute("SELECT id, email, total, status FROM pedidos ORDER BY id;").fetchall()

for fila in filas:
    print(dict(fila))


# 27. Insertar muchas filas con `executemany`

Cuando tenemos muchos registros, podemos usar `executemany`.

Recibe:

1. Una sentencia SQL con parámetros `?`.
2. Una lista de tuplas con los datos.


In [ ]:
pedidos_extra = [
    ("ana@example.com", 7.50, "paid", now_iso()),
    ("sofia@example.com", 32.10, "pending", now_iso()),
    ("david@example.com", 88.00, "paid", now_iso()),
]

with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.executemany(
        "INSERT INTO pedidos(email, total, status, created_at) VALUES (?, ?, ?, ?);",
        pedidos_extra
    )

    print("Filas insertadas:", cur.rowcount)


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row
    filas = conn.execute("""
        SELECT id, email, total, status
        FROM pedidos
        ORDER BY id;
    """).fetchall()

for fila in filas:
    print(dict(fila))


# 28. Inspeccionar una base de datos desde Python

Ahora que ya sabemos `SELECT`, `fetchone()` y `fetchall()`, tiene sentido aprender a inspeccionar la estructura de la base de datos.

SQLite guarda información interna en `sqlite_master`.

Esto permite:

- listar tablas;
- ver el SQL con el que se creó una tabla;
- comprobar columnas;
- depurar bases de datos que no hemos creado nosotros.


## 29. Listar tablas con `sqlite_master`

Consulta:

```sql
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
```


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;")
    filas = cur.fetchall()

print("Filas devueltas:", filas)

print("\nNombres de tablas:")
for fila in filas:
    print(fila[0])


## 30. Convertirlo en función, sin `return` comprimido


In [ ]:
def listar_tablas(conn):
    cur = conn.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name;")
    filas = cur.fetchall()

    tablas = []
    for fila in filas:
        nombre_tabla = fila[0]
        tablas.append(nombre_tabla)

    return tablas

with sqlite3.connect(DB_PATH) as conn:
    tablas = listar_tablas(conn)

print(tablas)


## 31. Ver columnas con `PRAGMA table_info`

`PRAGMA table_info(pedidos)` devuelve información de las columnas.

Cada fila tiene esta forma:

```text
(cid, name, type, notnull, dflt_value, pk)
```


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute("PRAGMA table_info(pedidos);")
    columnas = cur.fetchall()

for columna in columnas:
    print(columna)


## 32. Ver el SQL de creación de una tabla

Aquí esperamos una sola fila, así que usamos `fetchone()`.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute("""
        SELECT sql
        FROM sqlite_master
        WHERE type = 'table' AND name = ?;
    """, ("pedidos",))

    fila = cur.fetchone()

if fila is None:
    print("No se ha encontrado la tabla")
else:
    print(fila[0])


# 33. Crear una tabla con más validaciones

Ahora crearemos `pedidos_validados` con restricciones:

- `total >= 0`;
- `status` solo puede ser `paid`, `pending` o `cancelled`;
- `created_at` tiene valor por defecto.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute("DROP TABLE IF EXISTS pedidos_validados;")

    cur.execute("""
        CREATE TABLE pedidos_validados (
            id INTEGER PRIMARY KEY,
            email TEXT NOT NULL,
            total REAL NOT NULL CHECK(total >= 0),
            status TEXT NOT NULL CHECK(status IN ('paid', 'pending', 'cancelled')),
            created_at TEXT NOT NULL DEFAULT (datetime('now'))
        );
    """)

print("Tabla pedidos_validados creada")


## 34. Errores de integridad

Si incumplimos una restricción, SQLite lanza `sqlite3.IntegrityError`.


In [ ]:
try:
    with sqlite3.connect(DB_PATH) as conn:
        cur = conn.cursor()
        cur.execute(
            "INSERT INTO pedidos_validados(email, total, status) VALUES (?, ?, ?);",
            ("error@example.com", -10, "paid")
        )
except sqlite3.IntegrityError as e:
    print("IntegrityError:", e)


In [ ]:
try:
    with sqlite3.connect(DB_PATH) as conn:
        cur = conn.cursor()
        cur.execute(
            "INSERT INTO pedidos_validados(email, total, status) VALUES (?, ?, ?);",
            ("error@example.com", 20, "inventado")
        )
except sqlite3.IntegrityError as e:
    print("IntegrityError:", e)


# 35. Transacciones: todo o nada

Una transacción agrupa varias operaciones.

Con `with sqlite3.connect(...) as conn:`:

- si no hay error, se hace `commit`;
- si hay una excepción, se hace `rollback`.


In [ ]:
try:
    with sqlite3.connect(DB_PATH) as conn:
        cur = conn.cursor()

        cur.execute(
            "INSERT INTO pedidos_validados(email, total, status) VALUES (?, ?, ?);",
            ("ok@example.com", 10, "paid")
        )

        # Esta segunda inserción falla.
        cur.execute(
            "INSERT INTO pedidos_validados(email, total, status) VALUES (?, ?, ?);",
            ("mal@example.com", -5, "paid")
        )
except sqlite3.IntegrityError as e:
    print("Ha fallado la transacción:", e)

with sqlite3.connect(DB_PATH) as conn:
    filas = conn.execute("SELECT * FROM pedidos_validados;").fetchall()

print("Filas en pedidos_validados:", filas)


# 36. Primer modelo relacional: usuarios y pedidos

Hasta ahora `pedidos` guardaba el email directamente.

En un modelo más realista separaríamos:

- `usuarios`
- `pedidos_rel`

Un usuario puede tener muchos pedidos: relación **1 a N**.

```text
usuarios 1 ---- N pedidos_rel
```


## 37. Crear tablas relacionadas

Para que SQLite compruebe claves foráneas hay que activar:

```python
conn.execute("PRAGMA foreign_keys = ON;")
```


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()

    cur.execute("DROP TABLE IF EXISTS pedidos_rel;")
    cur.execute("DROP TABLE IF EXISTS usuarios;")

    cur.execute("""
        CREATE TABLE usuarios (
            email TEXT PRIMARY KEY,
            nombre TEXT NOT NULL,
            pais TEXT NOT NULL,
            created_at TEXT NOT NULL
        );
    """)

    cur.execute("""
        CREATE TABLE pedidos_rel (
            id INTEGER PRIMARY KEY,
            user_email TEXT NOT NULL,
            total REAL NOT NULL CHECK(total >= 0),
            status TEXT NOT NULL CHECK(status IN ('paid', 'pending', 'cancelled')),
            created_at TEXT NOT NULL,
            FOREIGN KEY (user_email) REFERENCES usuarios(email)
        );
    """)

print("Tablas usuarios y pedidos_rel creadas")


## 38. Insertar datos relacionados

Primero insertamos usuarios. Después insertamos pedidos asociados a esos usuarios.


In [ ]:
usuarios = [
    ("ana@example.com", "Ana", "España", now_iso()),
    ("luis@example.com", "Luis", "España", now_iso()),
    ("marta@example.com", "Marta", "Portugal", now_iso()),
    ("sofia@example.com", "Sofía", "Francia", now_iso()),
]

pedidos_rel = [
    ("ana@example.com", 19.99, "paid", now_iso()),
    ("ana@example.com", 7.50, "paid", now_iso()),
    ("luis@example.com", 55.00, "pending", now_iso()),
    ("marta@example.com", 120.50, "paid", now_iso()),
]

with sqlite3.connect(DB_PATH) as conn:
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()

    cur.executemany(
        "INSERT INTO usuarios(email, nombre, pais, created_at) VALUES (?, ?, ?, ?);",
        usuarios
    )

    cur.executemany(
        "INSERT INTO pedidos_rel(user_email, total, status, created_at) VALUES (?, ?, ?, ?);",
        pedidos_rel
    )

print("Datos relacionados insertados")


## 39. Consultar cada tabla por separado


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    print("Usuarios:")
    for fila in conn.execute("SELECT email, nombre, pais FROM usuarios ORDER BY email;"):
        print(fila)

    print("\nPedidos:")
    for fila in conn.execute("SELECT id, user_email, total, status FROM pedidos_rel ORDER BY id;"):
        print(fila)


# 40. Unir tablas con `JOIN`

Un `JOIN` combina información de varias tablas.

Queremos ver pedidos junto con el nombre del usuario:

```sql
FROM pedidos_rel p
JOIN usuarios u ON u.email = p.user_email
```


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row

    consulta = """
        SELECT
            p.id,
            u.nombre,
            u.email,
            p.total,
            p.status
        FROM pedidos_rel p
        JOIN usuarios u ON u.email = p.user_email
        ORDER BY p.id;
    """

    filas = conn.execute(consulta).fetchall()

for fila in filas:
    print(dict(fila))


## 41. `LEFT JOIN`: usuarios aunque no tengan pedidos

`LEFT JOIN` devuelve todas las filas de la tabla izquierda, aunque no haya coincidencia en la derecha.

Nos permite ver usuarios sin pedidos.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row

    consulta = """
        SELECT
            u.email,
            u.nombre,
            p.id AS pedido_id,
            p.total
        FROM usuarios u
        LEFT JOIN pedidos_rel p ON p.user_email = u.email
        ORDER BY u.email, p.id;
    """

    filas = conn.execute(consulta).fetchall()

for fila in filas:
    print(dict(fila))


# 42. Agrupar con `GROUP BY`

`GROUP BY` permite calcular resúmenes:

- cuántos pedidos tiene cada usuario;
- cuánto ha gastado cada usuario;
- cuántos usuarios hay por país.

Funciones habituales:

- `COUNT()` cuenta filas;
- `SUM()` suma valores;
- `AVG()` calcula medias.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    conn.row_factory = sqlite3.Row

    consulta = """
        SELECT
            u.email,
            u.nombre,
            COUNT(p.id) AS n_pedidos,
            COALESCE(SUM(p.total), 0) AS total_gastado
        FROM usuarios u
        LEFT JOIN pedidos_rel p ON p.user_email = u.email
        GROUP BY u.email, u.nombre
        ORDER BY total_gastado DESC;
    """

    filas = conn.execute(consulta).fetchall()

for fila in filas:
    print(dict(fila))


# 43. Índices

Un índice ayuda a encontrar datos más rápido, especialmente en columnas usadas en:

- `WHERE`
- `JOIN`
- `ORDER BY`

Pero no conviene crear índices sin criterio, porque ocupan espacio y pueden ralentizar escrituras.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    cur = conn.cursor()

    cur.execute("CREATE INDEX IF NOT EXISTS idx_pedidos_rel_user_email ON pedidos_rel(user_email);")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_pedidos_rel_status ON pedidos_rel(status);")

print("Índices creados")

## 44. Ver el plan con `EXPLAIN QUERY PLAN`

`EXPLAIN QUERY PLAN` muestra una explicación aproximada de cómo SQLite ejecutará una consulta.


In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    consulta = "SELECT * FROM pedidos_rel WHERE user_email = ?;"
    parametros = ("ana@example.com",)

    for fila in conn.execute("EXPLAIN QUERY PLAN " + consulta, parametros):
        print(fila)


# 45. Leer datos con Pandas

Pandas puede leer directamente una consulta SQL con `pd.read_sql_query`.

Esto es útil para análisis y reporting.


In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None

if pd is None:
    print("Pandas no está instalado en este entorno.")
else:
    with sqlite3.connect(DB_PATH) as conn:
        df = pd.read_sql_query("""
            SELECT
                u.email,
                u.nombre,
                u.pais,
                COUNT(p.id) AS n_pedidos,
                COALESCE(SUM(p.total), 0) AS total_gastado
            FROM usuarios u
            LEFT JOIN pedidos_rel p ON p.user_email = u.email
            GROUP BY u.email, u.nombre, u.pais
            ORDER BY total_gastado DESC;
        """, conn)

    display(df)


# 46. Buenas prácticas básicas

- Usa `with sqlite3.connect(...) as conn:`.
- Usa parámetros `?` para datos variables.
- No montes SQL con f-strings si intervienen datos externos.
- Usa `fetchone()` cuando esperas una fila.
- Usa `fetchall()` cuando quieres todas las filas.
- Comprueba si `fetchone()` devuelve `None`.
- Activa `PRAGMA foreign_keys = ON` cuando uses claves foráneas.
- Añade restricciones como `NOT NULL`, `CHECK` y `PRIMARY KEY`.
- Crea índices solo cuando tenga sentido.


# 47. Errores frecuentes

## Olvidar la coma en una tupla de un parámetro

Incorrecto:

```python
cur.execute("SELECT * FROM pedidos WHERE id = ?", (1))
```

Correcto:

```python
cur.execute("SELECT * FROM pedidos WHERE id = ?", (1,))
```

## Olvidar `WHERE` en `UPDATE` o `DELETE`

```sql
UPDATE pedidos SET status = 'paid';
```

Eso modifica todas las filas.

## Esperar que `execute` muestre resultados automáticamente

`execute` ejecuta la consulta, pero después hay que usar `fetchone`, `fetchall` o un `for`.


# 48. Mini práctica final

Crea una base de datos nueva llamada `biblioteca.db`.

## Parte 1: una tabla

1. Crea una tabla `libros`:
   - `id INTEGER PRIMARY KEY`
   - `titulo TEXT NOT NULL`
   - `autor TEXT NOT NULL`
   - `anio INTEGER`
   - `estado TEXT NOT NULL`

2. Inserta al menos 5 libros.

3. Consulta:
   - todos los libros;
   - libros de un autor concreto;
   - libros ordenados por año;
   - un libro concreto usando `fetchone()`.

## Parte 2: actualizar y borrar

4. Cambia el estado de un libro.
5. Borra un libro de prueba.

## Parte 3: relaciones

6. Crea una tabla `socios`.
7. Crea una tabla `prestamos` que relacione socios y libros.
8. Haz un `JOIN` para mostrar nombre del socio, título del libro y fecha del préstamo.

## Bonus

9. Cuenta cuántos préstamos tiene cada socio con `GROUP BY`.


# 49. Chuleta rápida

## Crear tabla

```sql
CREATE TABLE tabla (
    id INTEGER PRIMARY KEY,
    campo TEXT NOT NULL
);
```

## Insertar

```python
cur.execute(
    "INSERT INTO tabla(campo) VALUES (?);",
    (valor,)
)
```

## Consultar muchas filas

```python
cur.execute("SELECT * FROM tabla;")
filas = cur.fetchall()
```

## Consultar una fila

```python
cur.execute("SELECT * FROM tabla WHERE id = ?;", (id_buscado,))
fila = cur.fetchone()
```

## Actualizar

```python
cur.execute(
    "UPDATE tabla SET campo = ? WHERE id = ?;",
    (nuevo_valor, id_buscado)
)
```

## Borrar

```python
cur.execute(
    "DELETE FROM tabla WHERE id = ?;",
    (id_buscado,)
)
```
